# Taller Interactivo: Vectores, Sistemas de Fuerzas y Vector Resultante
### Física Mecánica — Preparatorio para Leyes de Newton

Este taller conecta dos momentos del curso donde los vectores son clave: la **velocidad** (cinemática, como en el movimiento parabólico) y las **fuerzas** (previo a las Leyes de Newton). La idea central que trabajaremos una y otra vez:

> Un objeto no se mueve hacia donde apunta *una* influencia, sino hacia la **suma vectorial** de todas. El bote no va hacia donde apunta el timón; va hacia donde lo lleva la combinación de timón, corriente y viento.

**Cómo usar este taller:** cada punto es autónomo e incluye un esquema, sliders para explorar, un botón **"Mostrar solución"** (oculta hasta que la pidas), un cuestionario que suma a tu nota, y un bloque **TIP-IA** para razonar con un asistente. Ejecuta `Celda → Ejecutar todo`. Si un slider no aparece, re-ejecuta esa celda.

**Notación:** resultados con tres cifras significativas y coma decimal (colombiana). Ángulos medidos desde el eje +x en sentido antihorario.

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material de uso académico. Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.

## Punto 1 — Descomposición e intuición: la lancha en el río  ★  [CL, UCM]
Una lancha cruza un río. El motor la impulsa en una dirección (el **rumbo** que apunta el timón), pero la **corriente** la arrastra río abajo. ¿Hacia dónde se mueve *realmente*? No hacia donde apunta el timón: hacia la **suma** de las dos velocidades.

Trabajaremos con velocidades (no fuerzas): la velocidad del bote respecto al agua más la velocidad de la corriente dan la velocidad respecto a tierra. Aquí es donde la mayoría construye —o rompe— su intuición vectorial.

- **Descomposición:** un vector de magnitud A e inclinación θ sobre +x tiene componentes Aₓ = A·cos θ, Ay = A·sen θ.
- **Suma:** las componentes se suman por separado: Rₓ = ΣAᵢₓ, Ry = ΣAᵢy.
- **Idea clave:** las componentes perpendiculares son **independientes**. La componente que cruza el río decide el *tiempo* de cruce; la corriente (paralela a la orilla) solo decide *cuánto río abajo* se llega.

> 🔧 **Aplicación en ingeniería:** en diseño de cruces fluviales, transporte de barcazas en el Magdalena o instalación de tuberías submarinas, se calcula el rumbo que debe apuntar una embarcación para compensar la corriente y llegar a un punto exacto de la otra orilla. Apuntar "derecho" casi nunca lleva al frente.

> 🧠 **Defiende tu conclusión:** distingue dos objetivos que se confunden: *cruzar en el menor tiempo* (apuntar perpendicular) y *llegar justo enfrente sin derivar* (apuntar contra la corriente). No son lo mismo, y la respuesta "correcta" depende de cuál persigues.

> 🤖 **TIP-IA — Pon a prueba tu intuición:**
> Antes de calcular, escríbele a un asistente tu predicción: *"Una lancha apunta perpendicular a la orilla a 4 m/s y la corriente es de 3 m/s. Yo creo que llegará [tu predicción: ¿enfrente? ¿río abajo? ¿cuánto?]. No me des la respuesta: hazme preguntas que me ayuden a verificar si mi predicción es consistente con la suma de vectores."* Primero razonas tú; la IA solo verifica tu razonamiento.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print_es(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass
# --- fin setup ---


def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.6))
    # IZQUIERDA: por qué el bote no va hacia donde apunta
    vb=np.array([0,4.0]); vc=np.array([3.0,0]); vr=vb+vc
    ax1.annotate('',xy=vb,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#2980b9',lw=2.5))
    ax1.annotate('',xy=vc,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#e67e22',lw=2.5))
    ax1.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    # completar el paralelogramo (regla de la suma)
    ax1.plot([vb[0],vr[0]],[vb[1],vr[1]],'--',color='#e67e22',lw=1,alpha=0.7)
    ax1.plot([vc[0],vr[0]],[vc[1],vr[1]],'--',color='#2980b9',lw=1,alpha=0.7)
    ax1.text(0.15,2.2,'v_bote\n(hacia donde\napunta el timón)',color='#2980b9',fontsize=9,fontweight='bold')
    ax1.text(1.4,-0.55,'v_corriente',color='#e67e22',fontsize=9,fontweight='bold')
    ax1.text(1.7,3.2,'v_real',color='#c0392b',fontsize=10,fontweight='bold')
    ax1.set_xlim(-1,5); ax1.set_ylim(-1.2,5); ax1.set_aspect('equal')
    ax1.axhline(0,color='#7f8c8d',lw=1); ax1.axvline(0,color='#7f8c8d',lw=0.5)
    ax1.set_title('El bote NO va hacia donde apunta:\nva hacia la SUMA de velocidades',fontsize=10)
    ax1.set_xticks([]); ax1.set_yticks([])
    # DERECHA: el río, la trayectoria real y la deriva
    W=60; t_cruce=W/vb[1]; deriva=vc[0]*t_cruce
    ax2.axhspan(0,W,color='#d6eaf8',alpha=0.6)
    ax2.plot([0,deriva],[0,W],color='#c0392b',lw=2.5,label=f'trayectoria real ({np.hypot(deriva,W):.0f} m)')
    ax2.plot([0,0],[0,W],'--',color='#2980b9',lw=1.5,label='rumbo apuntado (ideal)')
    ax2.scatter([0],[0],color='k',s=50,zorder=5); ax2.text(0.5,-4,'salida',fontsize=9)
    ax2.scatter([deriva],[W],color='#c0392b',s=60,zorder=5)
    ax2.text(deriva+1,W,f'llega {deriva:.0f} m\nrío abajo',fontsize=9,color='#c0392b')
    ax2.set_xlim(-8,55); ax2.set_ylim(-8,68); ax2.set_xlabel('deriva río abajo (m)'); ax2.set_ylabel('ancho del río (m)')
    ax2.set_title('Cruzar un río de 60 m con corriente',fontsize=10); ax2.legend(fontsize=8,loc='lower right')
    plt.tight_layout(); plt.show()
esquema()

def lancha(v_bote=4.0, rumbo=90.0, v_corr=3.0):
    # rumbo: ángulo del bote respecto a +x (90° = perpendicular, apuntando a cruzar)
    th=np.radians(rumbo)
    vb=np.array([v_bote*np.cos(th), v_bote*np.sin(th)])
    vc=np.array([v_corr,0.0])
    vr=vb+vc
    mag=np.hypot(*vr); ang=np.degrees(np.arctan2(vr[1],vr[0]))
    fig,ax=plt.subplots(figsize=(6.5,5))
    ax.annotate('',xy=vb,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#2980b9',lw=2.5))
    ax.annotate('',xy=(vc[0],vc[1]),xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#e67e22',lw=2.5))
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.plot([vb[0],vr[0]],[vb[1],vr[1]],'--',color='#e67e22',lw=1,alpha=0.6)
    ax.plot([vc[0],vr[0]],[vc[1],vr[1]],'--',color='#2980b9',lw=1,alpha=0.6)
    lim=max(6, mag+1)
    ax.set_xlim(-lim*0.4,lim); ax.set_ylim(-1.5,lim)
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=0.5); ax.set_aspect('equal')
    t1=f'|v_real|={mag:.3g} m/s'.replace('.',',')
    t2=f'rumbo real={ang:.3g}° sobre +x'.replace('.',',')
    desv=90-ang
    t3=f'desviación={desv:.3g}° río abajo'.replace('.',',')
    ax.set_title(f'{t1}   ·   {t2}\n{t3}',fontsize=10)
    ax.set_xlabel('x (m/s)'); ax.set_ylabel('y (m/s)')
    plt.tight_layout(); plt.show()

interact(lancha,
    v_bote=FloatSlider(value=4.0,min=1.0,max=8.0,step=0.5,description='v_bote (m/s)'),
    rumbo=FloatSlider(value=90.0,min=45.0,max=135.0,step=5.0,description='rumbo (°)'),
    v_corr=FloatSlider(value=3.0,min=0.0,max=6.0,step=0.5,description='v_corr (m/s)'));

def solucion():
    print_es("LANCHA CRUZANDO EL RÍO — suma de velocidades (NO de fuerzas):")
    print_es("   El bote apunta perpendicular (rumbo 90°) a 4,00 m/s; la corriente")
    print_es("   arrastra a 3,00 m/s río abajo (+x). Ancho del río: 60,0 m.")
    print_es("")
    vb=np.array([0.0,4.00]); vc=np.array([3.00,0.0]); vr=vb+vc
    mag=np.hypot(*vr); ang=np.degrees(np.arctan2(vr[1],vr[0]))
    print_es(f"   Paso 1 — Sumar componentes:")
    print_es(f"      v_real = v_bote + v_corriente = (0+3,00 ; 4,00+0) = (3,00 ; 4,00) m/s")
    print_es(f"   Paso 2 — Magnitud: |v_real| = √(3,00²+4,00²) = {mag:.3g} m/s  (triángulo 3-4-5)")
    print_es(f"   Paso 3 — Dirección: θ = atan2(4,00 ; 3,00) = {ang:.3g}° sobre +x")
    print_es(f"      → el bote se desvía {90-ang:.3g}° río abajo respecto a donde apunta.")
    print_es("")
    W=60.0; t=W/vb[1]; deriva=vc[0]*t
    print_es(f"   Paso 4 — Tiempo de cruce: SOLO depende de la componente que cruza (v_bote_y):")
    print_es(f"      t = ancho / v_bote_y = 60,0 / 4,00 = {t:.3g} s")
    print_es(f"      OJO: la corriente NO cambia el tiempo de cruce (las componentes")
    print_es(f"      perpendiculares son independientes). Esto vale porque el bote")
    print_es(f"      apunta perpendicular; si apuntara torcido, cambiaría.")
    print_es(f"   Paso 5 — Deriva río abajo: d = v_corr · t = 3,00 × {t:.3g} = {deriva:.3g} m")
    print_es(f"      Distancia real recorrida = |v_real|·t = {mag*t:.3g} m = √(60²+45²). ✓")
    print_es("")
    print_es("   CONTRAINTUITIVO: para llegar JUSTO enfrente (sin derivar), el bote")
    print_es("   debe apuntar hacia AGUAS ARRIBA, no perpendicular. Necesita una")
    print_es("   componente x que cancele la corriente: v_bote·cos(rumbo) = -3,00.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("El bote apunta perpendicular a 4,00 m/s y la corriente lo arrastra a 3,00 m/s. ¿Cuál es la rapidez REAL respecto a tierra (m/s)?",
    5.00,0.03,"m/s","|v|=√(3²+4²)=5,00 m/s. Es un triángulo 3-4-5.", id_preg="Vec_p1_q_1", peso=1.0)
quiz_numerico("¿Cuánto tarda en cruzar un río de 60,0 m si su componente perpendicular es 4,00 m/s? (s)",
    15.0,0.03,"s","t=60,0/4,00=15,0 s. La corriente no afecta el tiempo de cruce.", id_preg="Vec_p1_q_2", peso=1.0)
quiz_numerico("¿Cuántos metros río abajo (deriva) termina la lancha? (m)",
    45.0,0.04,"m","d=v_corr·t=3,00×15,0=45,0 m.", id_preg="Vec_p1_q_3", peso=1.0)
quiz_opcion_multiple("Si la corriente se DUPLICA (de 3 a 6 m/s), pero el bote sigue apuntando perpendicular a 4 m/s, ¿cambia el TIEMPO de cruce?",
    ["Sí, tarda el doble","Sí, tarda menos","No cambia: el tiempo de cruce solo depende de la componente perpendicular (4 m/s), no de la corriente","Depende del ancho"],2,
    "El tiempo de cruce = ancho/v_perpendicular. La corriente solo aumenta la deriva río abajo, no el tiempo. Componentes perpendiculares independientes.", id_preg="Vec_p1_q_4", peso=1.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice 'para cruzar más rápido, el bote debe apuntar un poco contra la corriente'. ¿Correcto o solo convincente?",
    ["Correcto: contrarrestar la corriente siempre ayuda",
     "Solo convincente: apuntar contra la corriente reduce la componente perpendicular (la que cruza), así que tarda MÁS en cruzar. Sirve para no derivar, no para cruzar rápido. Confunde 'llegar enfrente' con 'cruzar rápido'",
     "Correcto: es lo más eficiente en todo sentido",
     "Depende de la marca del motor"],1,
    "Para cruzar en el MENOR tiempo, conviene apuntar perpendicular (máxima componente que cruza). Apuntar contra la corriente sirve para anular la deriva, pero a costa de tardar más. Son objetivos distintos.", id_preg="Vec_p1_concep_5", peso=2.0)


## Punto 2 — Tensión en un nudo de cercha  ★★  [UCM, PRC]
En el nudo de una cercha para un puente concurren dos cables de acero: **F₁ = 12,0 kN** a 30,0° sobre +x, y **F₂ = 18,0 kN** a 135° sobre +x. Queremos la fuerza **resultante** sobre el nudo: su magnitud y su dirección.

El reto conceptual aquí es el **signo de las componentes**: F₂ está en el segundo cuadrante, así que su componente x es negativa. Ignorar ese signo es el error más común, y lleva a un ángulo final equivocado.

- Descompón cada fuerza: Fₓ = F·cos θ, Fy = F·sen θ (respetando el signo según el cuadrante).
- Suma componentes: Rₓ = ΣFₓ, Ry = ΣFy.
- Magnitud R = √(Rₓ² + Ry²); dirección θ = atan2(Ry, Rₓ), ajustada al cuadrante.

> 🔧 **Aplicación en ingeniería:** el análisis de cerchas (puentes, techos industriales, torres de transmisión) se basa en sumar fuerzas concurrentes en cada nudo. La resultante define si el nudo está cargado a tensión o compresión, y con qué magnitud debe dimensionarse el acero según la NSR-10.

> 🧠 **Defiende tu conclusión:** si tu ángulo sale en un cuadrante distinto al que muestra el dibujo de los vectores, algo falla en los signos. Debes poder justificar por qué la resultante cae donde cae, no solo reportar el número que arrojó la calculadora.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Sumé F₁=12 kN@30° y F₂=18 kN@135° y obtuve R=[tu magnitud] a [tu ángulo]. Actúa como evaluador escéptico: revisa si respeté el signo negativo de la componente x de F₂ y si mi ángulo final es consistente con el cuadrante. No me des la solución; dime dónde revisar."*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print_es(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    def dibujar(mag,ang,color,etq):
        r=np.radians(ang); v=np.array([mag*np.cos(r),mag*np.sin(r)])
        ax.annotate('',xy=v,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2.5))
        ax.text(v[0]*0.55,v[1]*0.55+0.6,etq,color=color,fontsize=9,fontweight='bold')
        return v
    v1=dibujar(12,30,'#2980b9','F₁=12 kN @30°')
    v2=dibujar(18,135,'#e67e22','F₂=18 kN @135°')
    vr=v1+v2
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.text(vr[0]-1,vr[1]+0.5,'R (resultante)',color='#c0392b',fontsize=10,fontweight='bold')
    lim=20
    ax.set_xlim(-16,14); ax.set_ylim(-2,20); ax.set_aspect('equal')
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1)
    ax.set_title('Nudo de cercha: dos cables de acero concurrentes',fontsize=10)
    ax.set_xlabel('x (kN)'); ax.set_ylabel('y (kN)')
    plt.tight_layout(); plt.show()
esquema()

def cercha(F1=12.0, a1=30.0, F2=18.0, a2=135.0):
    r1=np.radians(a1); r2=np.radians(a2)
    v1=np.array([F1*np.cos(r1),F1*np.sin(r1)])
    v2=np.array([F2*np.cos(r2),F2*np.sin(r2)])
    vr=v1+v2; mag=np.hypot(*vr); ang=np.degrees(np.arctan2(vr[1],vr[0]))
    if ang<0: ang+=360
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    ax.annotate('',xy=v1,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#2980b9',lw=2.5))
    ax.annotate('',xy=v2,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#e67e22',lw=2.5))
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.plot([v1[0],vr[0]],[v1[1],vr[1]],'--',color='#e67e22',lw=1,alpha=0.5)
    ax.plot([v2[0],vr[0]],[v2[1],vr[1]],'--',color='#2980b9',lw=1,alpha=0.5)
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1); ax.set_aspect('equal')
    allx=[v1[0],v2[0],vr[0],0]; ally=[v1[1],v2[1],vr[1],0]
    ax.set_xlim(min(allx)-3,max(allx)+3); ax.set_ylim(min(ally)-2,max(ally)+3)
    t1=f'R={mag:.3g} kN'.replace('.',','); t2=f'θ={ang:.3g}° (antihorario desde +x)'.replace('.',',')
    ax.set_title(f'{t1}   ·   {t2}',fontsize=10); ax.set_xlabel('x (kN)'); ax.set_ylabel('y (kN)')
    plt.tight_layout(); plt.show()

interact(cercha,
    F1=FloatSlider(value=12.0,min=5,max=25,step=1,description='F₁ (kN)'),
    a1=FloatSlider(value=30.0,min=0,max=90,step=5,description='α₁ (°)'),
    F2=FloatSlider(value=18.0,min=5,max=25,step=1,description='F₂ (kN)'),
    a2=FloatSlider(value=135.0,min=90,max=180,step=5,description='α₂ (°)'));

def solucion():
    print_es("NUDO DE CERCHA — F₁=12,0 kN @30°, F₂=18,0 kN @135°:")
    F1,a1,F2,a2=12.0,30.0,18.0,135.0
    r1=np.radians(a1); r2=np.radians(a2)
    F1x,F1y=F1*np.cos(r1),F1*np.sin(r1)
    F2x,F2y=F2*np.cos(r2),F2*np.sin(r2)
    print_es(f"   Paso 1 — Componentes:")
    print_es(f"      F₁: Fx=12,0·cos30°={F1x:.3g} kN,  Fy=12,0·sen30°={F1y:.3g} kN")
    print_es(f"      F₂: Fx=18,0·cos135°={F2x:.3g} kN,  Fy=18,0·sen135°={F2y:.3g} kN")
    print_es(f"      (F₂x es NEGATIVA: el 2º cuadrante tiene x<0, y>0)")
    Rx=F1x+F2x; Ry=F1y+F2y
    print_es(f"   Paso 2 — Sumar: Rx={Rx:.3g} kN,  Ry={Ry:.3g} kN")
    mag=np.hypot(Rx,Ry); ang=np.degrees(np.arctan2(Ry,Rx))
    if ang<0: ang+=360
    print_es(f"   Paso 3 — Magnitud: R=√(Rx²+Ry²)={mag:.3g} kN")
    print_es(f"      Dirección: θ=atan2(Ry,Rx)={ang:.3g}° (2º cuadrante, porque Rx<0 y Ry>0)")
    print_es(f"      Si usaras atan(|Ry/Rx|)={np.degrees(np.arctan(abs(Ry/Rx))):.3g}° obtendrías el ángulo")
    print_es(f"      MAL: hay que sumar 180°-ese valor para caer en el cuadrante correcto.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("F₂=18,0 kN a 135°. Calcule su componente x, F₂ₓ (kN).",
    -12.7,0.05,"kN","F₂ₓ=18,0·cos135°=-12,7 kN. Negativa porque está en el 2º cuadrante.", id_preg="Vec_p2_q_6", peso=2.0)
quiz_numerico("Con F₁=12,0 kN@30° y F₂=18,0 kN@135°, calcule Rx (kN).",
    -2.34,0.08,"kN","Rx=12,0cos30°+18,0cos135°=10,4-12,7=-2,34 kN.", id_preg="Vec_p2_q_7", peso=2.0)
quiz_numerico("Calcule la magnitud de la resultante R (kN).",
    18.9,0.04,"kN","R=√((-2,34)²+18,7²)=18,9 kN.", id_preg="Vec_p2_q_8", peso=2.0)
quiz_opcion_multiple("La resultante tiene Rx<0 y Ry>0. ¿En qué cuadrante está y cuál es su ángulo aproximado?",
    ["1º cuadrante, ~45°","2º cuadrante, ~97°","3º cuadrante, ~180°","4º cuadrante, ~-45°"],1,
    "Rx<0, Ry>0 → 2º cuadrante. θ=atan2(18,7 ; -2,34)≈97,1°.", id_preg="Vec_p2_q_9", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante calcula θ=atan(|Ry/Rx|)=82,9° y reporta ese ángulo. ¿Está bien?",
    ["Sí: el arcotangente da el ángulo correcto siempre",
     "Solo parece correcto: atan(|Ry/Rx|) da el ángulo AGUDO respecto al eje x, pero como Rx<0 la resultante está en el 2º cuadrante. El ángulo real es 180°-82,9°=97,1°. Ignorar el signo de las componentes es el error clásico",
     "Sí, si redondea a 83°",
     "No, debe ser negativo"],1,
    "El arcotangente simple pierde la información del cuadrante. Con Rx<0, Ry>0 hay que corregir a 180°-θ_agudo. Por eso atan2 (que usa los signos) es más seguro.", id_preg="Vec_p2_concep_10", peso=2.0)


## Punto 3 — Anclaje de un motor: tres fuerzas y equilibrio  ★★  [UCM, PRC, ULC]
El soporte de un motor recibe tres fuerzas concurrentes: **Fᴀ = 25,0 N** a 0°, **Fʙ = 40,0 N** a 60,0°, y **Fᴄ = 30,0 N** a 210° (tercer cuadrante). Construiremos una **tabla sumatoria** de componentes y hallaremos la fuerza neta.

Este punto es el puente hacia la **Primera Ley de Newton**: si el soporte estuviera en equilibrio, la suma de todas las fuerzas sería cero. Al calcular una resultante distinta de cero, descubrimos que hace falta una fuerza adicional para equilibrar.

- Organiza una tabla con las componentes x e y de cada fuerza (con sus signos).
- Fuerza neta: F_neta = (Rₓ î + Ry ĵ); magnitud y dirección como siempre.
- Equilibrio (1ª ley): ΣF = 0 ⟺ reposo o velocidad constante.

> 🔧 **Aplicación en ingeniería:** el diseño de anclajes de maquinaria (motores, bombas, compresores) exige que la fuerza neta sobre los pernos sea conocida para dimensionarlos. Si el equipo debe estar en reposo, los soportes aportan justo la fuerza que anula la resultante de las cargas.

> 🧠 **Defiende tu conclusión:** "equilibrar" no es "reforzar". La fuerza que equilibra un sistema va en dirección **opuesta** a la resultante, no en la misma. Debes poder argumentar por qué, no solo dar su magnitud.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Escribe: *"Tengo tres fuerzas sobre un motor dadas por magnitud y ángulo. Ayúdame a traducir entre representaciones: de (magnitud, ángulo) a componentes (î, ĵ), y de vuelta a (magnitud, ángulo) para la resultante. No resuelvas el problema; muéstrame cómo se pasa de una forma a otra con un ejemplo distinto al mío."*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print_es(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    datos=[(25,0,'#2980b9','Fᴀ=25 N @0°'),(40,60,'#e67e22','Fʙ=40 N @60°'),(30,210,'#8e44ad','Fᴄ=30 N @210°')]
    for mag,ang,color,etq in datos:
        r=np.radians(ang); v=np.array([mag*np.cos(r),mag*np.sin(r)])
        ax.annotate('',xy=v,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2.5))
        ax.text(v[0]*0.6,v[1]*0.6,etq,color=color,fontsize=8.5,fontweight='bold')
    ax.scatter([0],[0],color='k',s=80,zorder=5)
    ax.text(1,-4,'soporte del motor',fontsize=9)
    ax.set_xlim(-35,45); ax.set_ylim(-25,40); ax.set_aspect('equal')
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1)
    ax.set_title('Anclaje de motor: tres fuerzas concurrentes',fontsize=10)
    ax.set_xlabel('x (N)'); ax.set_ylabel('y (N)')
    plt.tight_layout(); plt.show()
esquema()

def motor(FA=25.0, FB=40.0, aB=60.0, FC=30.0, aC=210.0):
    def comp(m,a): r=np.radians(a); return np.array([m*np.cos(r),m*np.sin(r)])
    va=comp(FA,0); vb=comp(FB,aB); vc=comp(FC,aC)
    vr=va+vb+vc; mag=np.hypot(*vr); ang=np.degrees(np.arctan2(vr[1],vr[0]))
    if ang<0: ang+=360
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    for v,color in [(va,'#2980b9'),(vb,'#e67e22'),(vc,'#8e44ad')]:
        ax.annotate('',xy=v,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2))
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.text(vr[0],vr[1],' F_neta',color='#c0392b',fontsize=10,fontweight='bold')
    allx=[va[0],vb[0],vc[0],vr[0],0]; ally=[va[1],vb[1],vc[1],vr[1],0]
    ax.set_xlim(min(allx)-5,max(allx)+8); ax.set_ylim(min(ally)-5,max(ally)+8); ax.set_aspect('equal')
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1)
    t1=f'F_neta={mag:.3g} N'.replace('.',','); t2=f'θ={ang:.3g}°'.replace('.',',')
    ax.set_title(f'{t1}   ·   {t2}',fontsize=10); ax.set_xlabel('x (N)'); ax.set_ylabel('y (N)')
    plt.tight_layout(); plt.show()

interact(motor,
    FA=FloatSlider(value=25.0,min=10,max=50,step=1,description='Fᴀ (N)'),
    FB=FloatSlider(value=40.0,min=10,max=60,step=1,description='Fʙ (N)'),
    aB=FloatSlider(value=60.0,min=0,max=90,step=5,description='αʙ (°)'),
    FC=FloatSlider(value=30.0,min=10,max=50,step=1,description='Fᴄ (N)'),
    aC=FloatSlider(value=210.0,min=180,max=270,step=5,description='αᴄ (°)'));

def solucion():
    print_es("ANCLAJE DE MOTOR — Fᴀ=25,0 N@0°, Fʙ=40,0 N@60°, Fᴄ=30,0 N@210°:")
    def comp(m,a): r=np.radians(a); return m*np.cos(r),m*np.sin(r)
    ax_,ay=comp(25,0); bx,by=comp(40,60); cx,cy=comp(30,210)
    print_es("   Paso 1 — Tabla de componentes (N):")
    print_es(f"      Fᴀ:  x={ax_:.3g}    y={ay:.3g}")
    print_es(f"      Fʙ:  x={bx:.3g}   y={by:.3g}")
    print_es(f"      Fᴄ:  x={cx:.3g}  y={cy:.3g}   (3er cuadrante: ambas negativas)")
    Rx=ax_+bx+cx; Ry=ay+by+cy
    print_es(f"   Paso 2 — Sumas: Rx={Rx:.3g} N,  Ry={Ry:.3g} N")
    mag=np.hypot(Rx,Ry); ang=np.degrees(np.arctan2(Ry,Rx))
    print_es(f"   Paso 3 — F_neta=√(Rx²+Ry²)={mag:.3g} N,  θ={ang:.3g}° (1er cuadrante)")
    print_es("")
    print_es("   PUENTE A LA 1ª LEY DE NEWTON: si el motor está en equilibrio estático,")
    print_es(f"   la suma de TODAS las fuerzas debe ser CERO. Como F_neta={mag:.3g} N ≠ 0,")
    print_es(f"   el soporte NO está equilibrado: haría falta una 4ª fuerza de {mag:.3g} N")
    print_es(f"   en dirección OPUESTA (θ={ang:.3g}°+180°={(ang+180):.3g}°) para anularla.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Fʙ=40,0 N a 60°. Calcule su componente y, Fʙy (N).",
    34.6,0.05,"N","Fʙy=40,0·sen60°=34,6 N.", id_preg="Vec_p3_q_11", peso=2.0)
quiz_numerico("Sume las tres componentes x: Rx (N).",
    19.0,0.06,"N","Rx=25,0+40,0cos60°+30,0cos210°=25+20-26,0=19,0 N.", id_preg="Vec_p3_q_12", peso=2.0)
quiz_numerico("Calcule la magnitud de la fuerza neta (N).",
    27.3,0.05,"N","F_neta=√(19,0²+19,6²)=27,3 N.", id_preg="Vec_p3_q_13", peso=2.0)
quiz_opcion_multiple("Si el motor estuviera en equilibrio estático, ¿cuánto debería valer la fuerza neta?",
    ["Igual a la fuerza más grande","Cero: en equilibrio la suma vectorial de todas las fuerzas es nula","El promedio de las tres","27,3 N"],1,
    "1ª ley de Newton: en equilibrio (reposo o velocidad constante), ΣF=0. La resultante distinta de cero significa que hay aceleración.", id_preg="Vec_p3_q_14", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: para equilibrar el soporte, un estudiante propone una 4ª fuerza de 27,3 N en la MISMA dirección que F_neta. ¿Correcto?",
    ["Sí: agregar otra fuerza igual estabiliza",
     "Al contrario: la fuerza equilibrante debe ir en dirección OPUESTA (θ+180°) para CANCELAR la resultante. En la misma dirección la duplicaría. Equilibrar es anular, no reforzar",
     "Sí, si es más pequeña",
     "Da igual la dirección mientras sea 27,3 N"],1,
    "Para equilibrar (ΣF=0) la 4ª fuerza debe ser igual en magnitud y OPUESTA en dirección. En la misma dirección, la fuerza neta se duplicaría en vez de anularse.", id_preg="Vec_p3_concep_15", peso=2.0)


## Punto 4 — Remolque de una barrera: notación î ĵ y cuadrantes  ★★  [UCM, PRC, ULC]
Tres embarcaciones remolcan una barrera de contención de crudo. Dos fuerzas vienen ya en componentes: **F₁ = (4,00 î + 3,00 ĵ) kN** y **F₂ = (−2,00 î + 5,00 ĵ) kN**. La tercera se da por magnitud y ángulo: **F₃ = 6,00 kN** a 240°. Buscamos la tracción total en notación de vectores unitarios y la dirección real de desplazamiento.

El reto aquí es la **corrección de cuadrante**: la resultante tendrá Rₓ < 0 y Ry > 0 (segundo cuadrante), y el arcotangente simple da un ángulo engañoso que hay que corregir.

- Descompón F₃ (240° cae en el tercer cuadrante: ambas componentes negativas).
- Suma en notación î ĵ: R = (Rₓ î + Ry ĵ).
- Para el ángulo: si Rₓ < 0 y Ry > 0, el ángulo real es 180° − atan(|Ry/Rₓ|).

> 🔧 **Aplicación en ingeniería:** en operaciones de contención de derrames (como las del Magdalena Medio o la bahía de Cartagena) y en remolque marítimo, se combinan las tracciones de varias embarcaciones para dirigir una carga hacia un punto. La dirección resultante decide si la barrera se despliega bien o se desvía.

> 🧠 **Defiende tu conclusión:** dos personas pueden obtener la misma magnitud y direcciones distintas. Solo una es consistente con los signos de las componentes. Un vector es magnitud **y** dirección: acertar la magnitud no basta.

> 🤖 **TIP-IA — Generador de problemas:**
> Escribe: *"Ya resolví un problema de tres fuerzas con corrección de cuadrante. Invéntame un problema NUEVO donde la resultante caiga en el CUARTO cuadrante (Rₓ>0, Ry<0), con datos numéricos, pero NO lo resuelvas: solo plantéalo. Yo lo resuelvo y luego verificas mi corrección de ángulo."*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print_es(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass
# --- fin setup ---


def esquema():
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    v1=np.array([4.0,3.0]); v2=np.array([-2.0,5.0])
    r=np.radians(240); v3=np.array([6*np.cos(r),6*np.sin(r)])
    for v,color,etq in [(v1,'#2980b9','F₁=(4 ; 3) kN'),(v2,'#e67e22','F₂=(-2 ; 5) kN'),(v3,'#8e44ad','F₃=6 kN @240°')]:
        ax.annotate('',xy=v,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2.5))
        ax.text(v[0]*0.55,v[1]*0.55,etq,color=color,fontsize=8.5,fontweight='bold')
    vr=v1+v2+v3
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.text(vr[0]-1,vr[1]+0.4,'R_total',color='#c0392b',fontsize=10,fontweight='bold')
    ax.scatter([0],[0],color='k',s=70,zorder=5); ax.text(0.3,-0.8,'barrera',fontsize=9)
    ax.set_xlim(-8,7); ax.set_ylim(-6,9); ax.set_aspect('equal')
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1)
    ax.set_title('Remolque de barrera: tres embarcaciones',fontsize=10)
    ax.set_xlabel('x (kN)'); ax.set_ylabel('y (kN)')
    plt.tight_layout(); plt.show()
esquema()

def remolque(F3=6.0, a3=240.0):
    v1=np.array([4.0,3.0]); v2=np.array([-2.0,5.0])
    r=np.radians(a3); v3=np.array([F3*np.cos(r),F3*np.sin(r)])
    vr=v1+v2+v3; mag=np.hypot(*vr); ang=np.degrees(np.arctan2(vr[1],vr[0]))
    if ang<0: ang+=360
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    for v,color in [(v1,'#2980b9'),(v2,'#e67e22'),(v3,'#8e44ad')]:
        ax.annotate('',xy=v,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2))
    ax.annotate('',xy=vr,xytext=(0,0),arrowprops=dict(arrowstyle='->',color='#c0392b',lw=3))
    ax.text(vr[0],vr[1],' R',color='#c0392b',fontsize=11,fontweight='bold')
    allx=[v1[0],v2[0],v3[0],vr[0],0]; ally=[v1[1],v2[1],v3[1],vr[1],0]
    ax.set_xlim(min(allx)-2,max(allx)+2); ax.set_ylim(min(ally)-2,max(ally)+2); ax.set_aspect('equal')
    ax.axhline(0,color='#7f8c8d',lw=1); ax.axvline(0,color='#7f8c8d',lw=1)
    t1=f'R={mag:.3g} kN'.replace('.',','); t2=f'θ={ang:.3g}°'.replace('.',',')
    ax.set_title(f'{t1}   ·   {t2}',fontsize=10); ax.set_xlabel('x (kN)'); ax.set_ylabel('y (kN)')
    plt.tight_layout(); plt.show()

interact(remolque,
    F3=FloatSlider(value=6.0,min=2,max=10,step=0.5,description='F₃ (kN)'),
    a3=FloatSlider(value=240.0,min=180,max=300,step=5,description='α₃ (°)'));

def solucion():
    print_es("REMOLQUE DE BARRERA — F₁=(4,00 ; 3,00), F₂=(-2,00 ; 5,00) kN, F₃=6,00 kN@240°:")
    r=np.radians(240); F3x,F3y=6*np.cos(r),6*np.sin(r)
    print_es(f"   Paso 1 — Descomponer F₃:")
    print_es(f"      F₃ₓ=6,00·cos240°={F3x:.3g} kN,  F₃y=6,00·sen240°={F3y:.3g} kN")
    print_es(f"      (240° está en el 3er cuadrante: ambas componentes negativas)")
    Rx=4-2+F3x; Ry=3+5+F3y
    print_es(f"   Paso 2 — Sumar en notación î ĵ:")
    print_es(f"      Rₓ=4,00+(-2,00)+({F3x:.3g})={Rx:.3g} kN")
    print_es(f"      Ry=3,00+5,00+({F3y:.3g})={Ry:.3g} kN")
    print_es(f"      R = ({Rx:.3g} î + {Ry:.3g} ĵ) kN")
    mag=np.hypot(Rx,Ry); ang_agudo=np.degrees(np.arctan(abs(Ry/Rx)))
    ang=np.degrees(np.arctan2(Ry,Rx))
    if ang<0: ang+=360
    print_es(f"   Paso 3 — Magnitud: R=√(Rx²+Ry²)={mag:.3g} kN")
    print_es(f"      CORRECCIÓN DE CUADRANTE: Rx<0 y Ry>0 → 2º cuadrante.")
    print_es(f"      atan(|Ry/Rx|)={ang_agudo:.3g}° es el ángulo agudo; el real es")
    print_es(f"      180°-{ang_agudo:.3g}°={ang:.3g}°. Hacia allí se desplaza la barrera.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("F₃=6,00 kN a 240°. Calcule su componente y, F₃y (kN).",
    -5.20,0.05,"kN","F₃y=6,00·sen240°=-5,20 kN. Negativa (3er cuadrante).", id_preg="Vec_p4_q_16", peso=2.0)
quiz_numerico("Calcule Rx total en notación î ĵ (kN).",
    -1.00,0.10,"kN","Rx=4,00-2,00-3,00=-1,00 kN.", id_preg="Vec_p4_q_17", peso=2.0)
quiz_numerico("Calcule la magnitud de la resultante total R (kN).",
    2.98,0.05,"kN","R=√((-1,00)²+2,80²)=2,98 kN.", id_preg="Vec_p4_q_18", peso=2.0)
quiz_opcion_multiple("R tiene Rx=-1,00 y Ry=+2,80. El arcotangente simple da 70,3°. ¿Cuál es la dirección REAL?",
    ["70,3° (1er cuadrante)","109,6° (2º cuadrante): como Rx<0, Ry>0, se corrige 180°-70,3°","-70,3°","250,3°"],1,
    "Rx<0, Ry>0 → 2º cuadrante. El ángulo real es 180°-70,3°=109,6°. El arcotangente simple ignora el signo de Rx.", id_preg="Vec_p4_q_19", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: dos estudiantes obtienen la misma magnitud R=2,98 kN pero uno reporta 70,3° y otro 109,6°. ¿Pueden ambos tener razón?",
    ["Sí: la magnitud es lo que importa, el ángulo es secundario",
     "No: solo 109,6° es correcto. Misma magnitud NO implica misma dirección; 70,3° apunta al 1er cuadrante (donde Rx>0), contradiciendo que Rx=-1,00<0. La dirección importa tanto como la magnitud en un vector",
     "Sí: 70,3° y 109,6° son equivalentes",
     "Depende de cómo se mida"],1,
    "Un vector queda definido por magnitud Y dirección. 70,3° ubicaría la resultante en el 1er cuadrante, incompatible con Rx negativo. Solo 109,6° es consistente con las componentes.", id_preg="Vec_p4_concep_20", peso=2.0)


---
## Calificación del taller
Pulsa el botón para ver tu nota formativa (0,0–5,0), ponderada por la dificultad de cada pregunta. Puedes reintentar los cuestionarios y recalcular.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print_es(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print_es("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass
# --- fin setup ---


calificacion_final(total_preguntas=20)